In [ ]:
import sys, os, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "flask", "flask-cors", "pyngrok", "openai", "requests",
    "--quiet", "--no-deps", "--upgrade"])
print(f"Python: {sys.executable}")
print("Setup complete")

In [ ]:
# ============================================================
#  AGENT COLISEUM -- BCP -- Colab 02: LangChain-style Agent
# ============================================================
import os, sys, json, random, re
from openai import OpenAI

import os, sys

# Ensure agent_base and agent_server are importable from this directory
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.abspath(".."))

# Load credentials from ~/.env_coliseum
_env = os.path.expanduser("~/.env_coliseum")
if os.path.exists(_env):
    for _l in open(_env):
        _l = _l.strip()
        if _l and "=" in _l:
            _k, _v = _l.split("=", 1)
            os.environ[_k] = _v
    print("Credentials loaded from ~/.env_coliseum")
else:
    print("WARNING: ~/.env_coliseum not found")

AZURE_API_KEY  = os.environ.get("AZURE_API_KEY",  "")
AZURE_BASE_URL = os.environ.get("AZURE_BASE_URL", "https://rsgd15-foundry.openai.azure.com/openai/v1/")
MODEL          = "gpt-5"
ARENA_URL      = "https://agent-coliseum.onrender.com"
NGROK_TOKEN    = os.environ.get("NGROK_TOKEN", "")

print(f"AZURE_API_KEY:  ...{AZURE_API_KEY[-6:] if AZURE_API_KEY else 'NOT SET'}")
print(f"AZURE_BASE_URL: {AZURE_BASE_URL}")
print(f"NGROK_TOKEN:    ...{NGROK_TOKEN[-6:] if NGROK_TOKEN else 'NOT SET'}")

from agent_base import Agent, MatchContext, MatchResult, WorldContext, Position
from agent_server import serve_and_register

client = OpenAI(base_url=AZURE_BASE_URL, api_key=AZURE_API_KEY)
print(f"Client ready: {client.base_url}")

SYSTEM_PROMPT = """You are a competitive Latin America knowledge agent.
Reason step by step. Write concrete responses under each label (do NOT leave blanks):

SITUATION [Wei et al. 2022 -- Chain-of-Thought]:
OPPONENT [Langner et al. 2023 -- Theory of Mind]:
GOAL [Yao et al. 2022 -- ReAct]:
DRAFT [Nye et al. 2021 -- Scratchpad]:
CRITIQUE [Madaan et al. 2023 -- Self-Refine]:
FINAL [Shinn et al. 2023 -- Reflexion]: <your question or answer, concise>"""

def think_chain_invoke(input_text: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": input_text},
        ],
        max_completion_tokens=2000,
    )
    return resp.choices[0].message.content.strip()

class LangChainLatAmAgent(Agent):
    name        = "LangChain Puma"
    avatar      = "🐆"
    description = "Agente con memoria de conversacion y razonamiento estructurado"

    def __init__(self):
        self._match_memory   = {}
        self._opponent_notes = {}

    def on_arena_start(self, ctx: WorldContext) -> None:
        self._match_memory   = {}
        self._opponent_notes = {}

    def on_match_start(self, ctx: MatchContext) -> None:
        self._match_memory[ctx.match_id] = []

    def on_match_end(self, ctx: MatchContext, result: MatchResult) -> None:
        won = result.winner_id == ctx.my_agent_id
        self._opponent_notes[ctx.opponent_agent_id] = {
            "name": ctx.opponent_name, "result": "won" if won else "lost", "topic": ctx.topic}

    def move(self, ctx: WorldContext) -> Position:
        dx, dy = random.choice([(0,1),(0,-1),(1,0),(-1,0),(0,0)])
        return Position(x=max(0, min(ctx.map_width-1,  ctx.my_position.x+dx)),
                        y=max(0, min(ctx.map_height-1, ctx.my_position.y+dy)))

    def think(self, ctx: MatchContext) -> str:
        history = ""
        for t in ctx.history[-3:]:
            history += f"\n  Turn {t['turn_number']}: Q={t['question'][:50]} A={t['answer'][:50]} Score={t['score']}"
        opp     = self._opponent_notes.get(ctx.opponent_agent_id, {})
        opp_txt = (f"Previously: {opp.get('result','unknown')} on {opp.get('topic','?')}"
                   if opp else "First encounter.")
        mem     = self._match_memory.get(ctx.match_id, [])
        mem_txt = "".join(f"\n  Turn {e['turn']} ({e['role']}): {e['summary']}" for e in mem[-2:])

        role_instruction = (
            f"Generate a sharp, specific question about {ctx.topic} that will be hard for {ctx.opponent_name} to answer."
            if ctx.role == "asker" else
            f"Answer this question accurately in 1-2 sentences: {ctx.current_question}"
        )

        input_text = f"""Topic: {ctx.topic} | Role: {ctx.role} | Turn {ctx.turn}/{ctx.total_turns}
My score: {sum(ctx.my_scores)} pts | Opponent: {sum(ctx.opponent_scores)} pts | Opponent: {ctx.opponent_name}
Opponent history: {opp_txt}
Match history:{history if history else " (first turn)"}
Memory:{mem_txt if mem_txt else " (empty)"}

Task: {role_instruction}"""

        result = think_chain_invoke(input_text)
        mem.append({"turn": ctx.turn, "role": ctx.role, "summary": result[:150]})
        self._match_memory[ctx.match_id] = mem
        return result

    def ask(self, ctx: MatchContext) -> str:
        return self._extract_final(self.think(ctx))

    def answer(self, ctx: MatchContext) -> str:
        return self._extract_final(self.think(ctx))

    def _extract_final(self, text: str) -> str:
        import re
        lines = text.split("\n")
        for i, line in enumerate(lines):
            clean = line.replace("**", "").strip()
            if clean.upper().startswith("FINAL"):
                after = clean[5:].lstrip(":").strip()
                after = re.sub(r"^\[.*?\]\s*:?\s*", "", after).strip()
                if after:
                    return after
                for j in range(i+1, len(lines)):
                    if lines[j].strip():
                        return lines[j].strip()
        for line in reversed(lines):
            if line.strip() and not line.strip().startswith("#"):
                return line.strip()
        return text.strip()

import threading, requests as _req, time as _time

def _keepalive(arena_url):
    while True:
        _time.sleep(60)
        try:
            _req.get(f"{arena_url}/health", timeout=5)
        except:
            pass

threading.Thread(target=_keepalive, args=(ARENA_URL,), daemon=True).start()
print("Keepalive thread started")

agent = LangChainLatAmAgent()

In [ ]:
# ── DEBUG CELL — test agent before connecting to arena ────────────────────
import requests, json as _json
from agent_base import MatchContext

# ── Test 1: Flask health ──────────────────────────────────────────────────
try:
    r = requests.get("http://localhost:5000/health", timeout=3)
    print("Flask health:", r.status_code, r.json())
except Exception as e:
    print("Flask NOT reachable:", e)

# ── Test 2: ngrok tunnel ──────────────────────────────────────────────────
try:
    r = requests.get("http://localhost:4040/api/tunnels", timeout=3)
    tunnels = r.json().get("tunnels", [])
    for t in tunnels:
        print("Ngrok tunnel:", t["public_url"], "->", t["config"]["addr"])
    if not tunnels:
        print("No active ngrok tunnels")
except Exception as e:
    print("Ngrok NOT reachable:", e)

# ── Test 3: raw think() output ────────────────────────────────────────────
test_ctx = MatchContext(
    match_id="test", topic="Latin American geography",
    turn=1, total_turns=3, role="asker",
    history=[], my_agent_id="me", opponent_agent_id="opp",
    opponent_name="Test Opponent", my_scores=[], opponent_scores=[],
)
print("\n=== RAW think() OUTPUT (asker) ===")
raw = agent.think(test_ctx)
print(raw)
print("\n=== EXTRACTED FINAL (asker) ===")
print(repr(agent._extract_final(raw)))

test_ctx2 = MatchContext(
    match_id="test", topic="Latin American geography",
    turn=1, total_turns=3, role="answerer",
    history=[], my_agent_id="me", opponent_agent_id="opp",
    opponent_name="Test Opponent", my_scores=[], opponent_scores=[],
    current_question="What is the longest river in South America?",
)
print("\n=== RAW think() OUTPUT (answerer) ===")
raw2 = agent.think(test_ctx2)
print(raw2)
print("\n=== EXTRACTED FINAL (answerer) ===")
print(repr(agent._extract_final(raw2)))

# ── Test 4: simulate /ask and /answer HTTP calls ──────────────────────────
payload = {
    "match_id": "test", "topic": "Latin American geography",
    "turn": 1, "total_turns": 3, "role": "asker",
    "history": [], "my_agent_id": "me", "opponent_agent_id": "opp",
    "opponent_name": "Test", "my_scores": [], "opponent_scores": [],
    "scratchpad": "", "current_question": ""
}
try:
    r = requests.post("http://localhost:5000/ask", json=payload, timeout=60)
    print("\n=== /ask HTTP response ===")
    print("Status:", r.status_code)
    print("Body:", _json.dumps(r.json(), indent=2))
except Exception as e:
    print("\n/ask call failed:", e)

payload["role"] = "answerer"
payload["current_question"] = "What is the longest river in South America?"
try:
    r = requests.post("http://localhost:5000/answer", json=payload, timeout=60)
    print("\n=== /answer HTTP response ===")
    print("Status:", r.status_code)
    print("Body:", _json.dumps(r.json(), indent=2))
except Exception as e:
    print("\n/answer call failed:", e)

In [ ]:
import subprocess
subprocess.run(["fuser", "-k", "5001/tcp"], capture_output=True)

from pyngrok import ngrok
ngrok.kill()

serve_and_register(
    agent       = agent,
    arena_url   = ARENA_URL,
    port        = 5001,
    ngrok_token = NGROK_TOKEN,
)